## Notebook for preliminaries

#### Attaches databases by default
#### Assembles the flagged table of sources

In [108]:
%run common_setup.ipynb

In [109]:
#### Source extraction

## Extract unique sources from JCR download
## Match to OpenAlex
## Flag Domingo journals

In [110]:
class SourcePreliminaries(SetUp):

    def __init__(self):
        super().__init__()
        return

    def extract_jcr(self):
        # The JCR is in excel parts reflecting the subjects in the Economics & Business area
        # Download, ensure that there is an ISSN, combine, and load into duckdb in memory
        df_list = []
        df_list.extend(pd.read_excel(file, skiprows=2) for file in glob.glob('../DATA/WOS_JCR/*.xlsx'))
        df = pd.concat(df_list).drop_duplicates()
        print(f'{df.shape = }\n{df.head()}')
        df.to_csv('../DATA/wos_jcr.csv', index=False)
        self.db.sql("""CREATE OR REPLACE TABLE memory.jcr_wos AS (SELECT DISTINCT "Journal name" AS jcr_name, ISSN, eISSN, Category  FROM df)""")
        self.db.sql("SELECT * FROM memory.jcr_wos").show()
        return

    def match_jcr_to_oa(self):
        # Match JCR to OpenAlex, using the ISSN and eISSN
        sql = """
            CREATE OR REPLACE TABLE memory.jcr_oa AS
                SELECT DISTINCT jcr_name, issn_l, j.ISSN, j.eISSN, id AS source_id, Category
                FROM memory.jcr_wos j
                LEFT JOIN sources.sources s
                ON list_contains(s.issn, j.ISSN) OR list_contains(s.issn, j.eISSN)
                WHERE id IS NOT NULL
                ORDER BY jcr_name
        """
        self.db.sql(sql)
        self.db.sql("SELECT * FROM memory.jcr_oa").show()
        self.db.sql("""SELECT count() AS item_count, 
                              count(DISTINCT jcr_name) AS distinct_jcr_count, 
                              count(DISTINCT source_id) AS distinct_source_count FROM memory.jcr_oa"""
                    ).show()
        return

    def extract_domingo_journals(self):
        # Extract journals or institutions from Domingo and load into duckdb in memory
        # The only information is the name. There are around 600 of each.
        # For journals, hopefully we can use the WOS JCR to map names to ISSN
        # For institutions we use the MatchMaker()
        df = pd.read_excel('../DATA/eco_bus_inst_journal_scores.xlsx', sheet_name='journals')
        df = self._tidy_journals(df=df)
        df = df.drop(columns=['Unnamed: 0'])
        print(f'{df.shape = }\n{df.head()}')
        self.db.sql("CREATE OR REPLACE TABLE memory.journals_domingo AS (SELECT * FROM df)")
        self.db.sql("SELECT * FROM memory.journals_domingo").show()
        self.db.sql("SELECT count(*) AS domingo_journal_count FROM memory.journals_domingo").show()
        return
        
    def match_domingo_journals(self):
        # Match Domingo journals to JCR names, and then match ISSN to OA  
        self._special_issn()  
        # Load special ISSN into memory
        # Match JCR names to Domingo names, and then match ISSN to OA
        # We will use the ISSN and eISSN from JCR, and the source_id from OA
        # We will also add a special case for 'International Journal of Arts Management'
        sql = """
            CREATE OR REPLACE TABLE project.sources_matched AS
            SELECT DISTINCT ON (journal)
                    sub.jcr_name,
                    issn_l,
                    CASE WHEN id NOT NULL THEN id 
                         WHEN sub.jcr_name = 'International Journal of Arts Management' THEN 'https://openalex.org/S4363607656' 
                         ELSE NULL END AS source_id,
                    journal,
                    acr,
                    pub,
                    journal_score
                FROM memory.journals_domingo j
                LEFT JOIN 
                    (SELECT jcr_name,
                            ISSN, eISSN,
                        FROM memory.jcr_oa
                     UNION
                     SELECT jcr_name,
                            ISSN, NULL AS eISSN
                        FROM memory.specials s
                    ) sub
                ON lower(journal) = lower(sub.jcr_name)
                LEFT JOIN sources.sources s
                ON list_contains(s.issn, sub.ISSN) = true OR list_contains(s.issn, sub.eISSN) OR display_name = journal
                -- WHERE source_id IS NOT NULL AND ACR != 'HBP' AND ACR != 'DEM'
                ORDER BY journal
            """
        self.db.sql(sql)
        self.db.sql("SELECT count(*) AS count_matched FROM project.sources_matched").show()
        df = self.db.sql("SELECT source_id, issn_l, journal, acr, pub, journal_score FROM project.sources_matched").df()
        print(f'NOT FOUND or DUPLICATED\n{df[df.duplicated(subset=["source_id"], keep=False)]}')

        return

    def _special_issn(self):
        special_issn = {'economic research-ekonomska istrazivanja': ['1331-677X'], # '1848-9664'],
                        'journal of the knowledge economy': 	['1868-7865'], #, '1868-7873'],
                        'latin american economic review': 	['2196-436X']} #, '2198-3526']}
        df = pd.DataFrame.from_dict(special_issn, orient='index').reset_index()
        df.columns = ['jcr_name', 'ISSN']
        print(f'SPECIAL matches {df.shape = }\n{df.head()}')
        self.db.sql("CREATE OR REPLACE TABLE memory.specials AS (SELECT * FROM df)")
        self.db.sql("SELECT count(*) FROM memory.specials").show()
        return

    def _tidy_journals(self, df=None):
        df = df.replace(to_replace='(?i)^Economics and Philosophy$', value='economics & philosophy', regex=True) 
        df = df.replace(to_replace='(?i)^revista de historia economica$', value='Revista de Historia Economica-Journal of Iberian and Latin American Economic History', regex=True)
        df = df.replace(to_replace='(?i)^spanish journal of finance and accounting-revista espanola de financiacion y contabilida$', value=
                                 'spanish journal of finance and accounting-revista espanola de financiacion y contabilidad', regex=True)
        df = df.replace(to_replace='(?i)BUSINESS ETHICS-A EUROPEAN REVIEW', value='BUSINESS ETHICS THE ENVIRONMENT & RESPONSIBILITY'.capitalize(), regex=True)
        return df

In [111]:
def main():

    sp = SourcePreliminaries()
    sp.extract_jcr() 
    sp.match_jcr_to_oa()
    sp.extract_domingo_journals() 
    sp.match_domingo_journals()
    # sp.special_issn()

In [112]:
if __name__ == '__main__':
    main()
    print("DONE!")

┌──────────────┬─────────┬──────────────────────┬──────────────────────┬───────────────────────────────────┬───────────┐
│   database   │ schema  │         name         │     column_names     │           column_types            │ temporary │
│   varchar    │ varchar │       varchar        │      varchar[]       │             varchar[]             │  boolean  │
├──────────────┼─────────┼──────────────────────┼──────────────────────┼───────────────────────────────────┼───────────┤
│ authors      │ main    │ authors              │ [id, orcid, displa…  │ [VARCHAR, VARCHAR, VARCHAR, 'VA…  │ false     │
│ institutions │ main    │ institutions         │ [id, ror, display_…  │ [VARCHAR, VARCHAR, VARCHAR, VAR…  │ false     │
│ institutions │ main    │ ror                  │ [name, institution…  │ [VARCHAR, VARCHAR]                │ false     │
│ project      │ main    │ author_citation_re…  │ [dupes, Research_P…  │ [BIGINT, VARCHAR, VARCHAR, BIGI…  │ false     │
│ project      │ main    │ autho